# Project DarkSnail V2 + HRM/TRM + Amygdala + Gemma 4 31B — ARC-AGI-3 Interactive Reasoning Agent Suite

**Architecture:** Bi-Hemispheric `DarkSnailArc3Agent` combining:
- **Right-Hemisphere Spatial-Temporal World Model (`DarkSnailArc3WorldModel` & `Arc3EpisodicBeltramiShell`):**
  1. **`TemporalClockContentSplitter` ($\mathcal{H}_{\text{clock}} \oplus \mathcal{H}_{\text{nope}}$) & Cross-Level Clock Persistence:** Exogenous-Endogenous MDP Factorization (`arXiv:1806.01584`, `arXiv:2207.08229`, `arXiv:1811.01483`) and discrete continuity equation $\partial_t \rho_c(\mathbf{x}, t) + \nabla \cdot \mathbf{J}_c(\mathbf{x}, t) = \sigma_c(\mathbf{x}, t)$, isolating exogenous timer/HUD depletion bars from controllable state with **zero spatial border priors**, 4-channel medoid masking (`fg_comps`, `enclosed_hole`, `symmetry_defect`, `rope2d_patch`), and `preserve_clock_mask=True` across levels.
  2. **Non-Local Causal Operator Factorization ($\mathcal{G}_{\text{act}}$ vs. $\Omega_{\text{reg}}$) & Dihedral $D_4$ Shape Transfer:** Distinguishes active peripheral actuators/sliders ($\eta_{\text{nonlocal}} \ge 0.50$ at Chebyshev distance $> 4$, stored in `remote_actuators`) from passive driven state registers (`driven_register_counts`, pruned via `is_driven_passive_register`), paired with rotation-invariant Dihedral $D_4$ shape transfer (`get_d4_shape_prior`).
  3. **`Arc3HRMRecurrentCore` + `SlowFastTriadCascade` + `SkewSymmetricTriadOperator`:** Self-contained Sapient Hierarchical Reasoning Model (`arXiv:2506.21734`) with `MagicNorm`, fast tactical `L_module`, slow strategic `H_module`, and Adaptive Computation Time `q_head` (`[Q_halt, Q_continue]`), coupled with exact `0.000e+00` spherical norm-conserving `SlowFastTriadCascade` and pointwise orthogonal ($\le 10^{-8}$) $\mathfrak{so}(8)$ action-triad operators.
  4. **`Arc3AffectiveAmygdalaRouter` (`[V_t, U_t, \Omega_t]`):** 3D affective state classifier (Valence $V_t \in [-1, 1]$, Epistemic Uncertainty $U_t \in [0, 1]$, Survival Urgency $\Omega_t \in [0, 1]$) driving continuous callosal E/I neuromodulatory gains $(\gamma_E, \gamma_I)$, sampling temperature $T_{\text{gen}}$, HRM unroll depth $N_{\text{steps}} \in [1, 8]$, and System-1 Reflex Bypass vs. System-2 escalation.
- **Left-Hemisphere Dual-Pathway System-2 Deliberation (`google/gemma-4-31B-it` + `Gemma4BeltramiCallosalShell`):**
  - Frozen `google/gemma-4-31B-it` ($L=60$, $d=5376$, 10 Global Full-Attention layers `[5, 11, 17, 23, 29, 35, 41, 47, 53, 59]` patched with `SurgicalGemma4Attention` and `weights/gemma4_31b_router_v4.safetensors`).
  - Event-triggered `deliberate_with_gemma4_system2` executes **Pathway A** (Global Layer-29 `Gemma4BeltramiCallosalShell` `(1, T, 5376) -> (1, 32, 4096) -> (1, 32, 512)` into `z_H_super`) and **Pathway B** (next-token digit `"0".."9"` logit softmax into `gemma4_action_priors`).

### Notebook Structure:
1. **Cell 1 — Environment Setup, Repo Sync (`sneed-and-feed/projectdarksnail`), Offline 25-Game Suite & Imports**
2. **Cell 2 — Mathematical Invariants Gate (`TemporalClockContentSplitter`, $\mathcal{G}_{\text{act}} / \Omega_{\text{reg}}$ Factorization, `Arc3HRMRecurrentCore`, `Arc3AffectiveAmygdalaRouter` & `SlowFastTriadCascade`)**
3. **Cell 3 — Live `google/gemma-4-31B-it` + `gemma4_31b_router_v4.safetensors` + Dual-Pathway Layer-29 `attach_gemma4_system2` Integration**
4. **Cell 4 — Interactive Single-Game Telemetry (`ls20` Fuel-Clock Navigation, `vc33` Hydraulic Puzzle, `sc25` Multi-Level Actuator & `lp85` Stepper)**
5. **Cell 5 — Full 25-Game ARC-AGI-3 Offline Evaluation Suite (`random_legal` vs. `unshielded_greedy` vs. `darksnail_v2_arc3`)**
6. **Cell 6 — Visual Diagnostic Gallery ($\mathcal{H}_{\text{clock}}$ Masking & Dyadic Quadtree Medoids)**

## 1. Environment Setup, Repository Sync & Offline 25-Game Suite Verification

In [ ]:
# === CELL 1: Environment Setup, Repository Sync & ARC-AGI-3 Offline Suite ===
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import torch

# 1. Load HF_TOKEN from Google Colab Secrets (or environment fallback)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("[OK] Loaded HF_TOKEN from Google Colab Secrets.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print("[OK] Loaded HF_TOKEN from local environment.")
    else:
        print("[INFO] HF_TOKEN not found; Right-Hemisphere DarkSnailArc3Agent runs standalone if no LLM token is present.")

# 2. Clone or update https://github.com/sneed-and-feed/projectdarksnail.git if running inside Google Colab
TARGET_BRANCH = os.environ.get("DARKSNAIL_BRANCH", "main")
if "google.colab" in sys.modules:
    repo_dir = "/content/darksnail"
    if not os.path.exists(f"{repo_dir}/.git"):
        subprocess.run(["git", "clone", "https://github.com/sneed-and-feed/projectdarksnail.git", repo_dir], check=False)
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", "--all"], check=False)
    subprocess.run(["git", "-C", repo_dir, "checkout", TARGET_BRANCH], check=False)
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)

    required_pkgs = [
        "arc-agi",
        "arcengine",
        "safetensors",
        "einops",
        "huggingface_hub",
        "transformers",
        "accelerate",
        "scipy",
        "bitsandbytes",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *required_pkgs], check=False)

    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
        if repo_dir in sys.path:
            sys.path.remove(repo_dir)
        sys.path.insert(0, repo_dir)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    cwd = os.getcwd()
    if cwd not in sys.path:
        sys.path.insert(0, cwd)

# 3. Ensure the 25 public ARC-AGI-3 offline environments exist in data/arc_agi_3_environments
env_dir = Path("data/arc_agi_3_environments")
PUBLIC_25_GAMES = [
    "ar25", "bp35", "cd82", "cn04", "dc22", "ft09", "g50t", "ka59", "lf52",
    "lp85", "ls20", "m0r0", "r11l", "re86", "s5i5", "sb26", "sc25", "sk48",
    "sp80", "su15", "tn36", "tr87", "tu93", "vc33", "wa30",
]
if not env_dir.exists() or len(list(env_dir.iterdir())) < 25:
    print("[INFO] Downloading 25 public ARC-AGI-3 environments into data/arc_agi_3_environments...")
    from arc_agi import Arcade, OperationMode
    env_dir.mkdir(parents=True, exist_ok=True)
    dl_arc = Arcade(operation_mode=OperationMode.NORMAL, environments_dir=str(env_dir))
    for gid in PUBLIC_25_GAMES:
        try:
            dl_arc.make(gid)
        except Exception as exc:
            print(f"  [WARN] Could not download {gid}: {exc}")

device = "cuda" if torch.cuda.is_available() else "cpu"
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024**3)) if device == "cuda" else 0.0
print(f"Runtime Device: {device.upper()}" + (f" ({torch.cuda.get_device_name(0)}, {vram_gb:.1f} GB VRAM)" if device == "cuda" else " (CPU)"))

import darksnail
import darksnail.arc3
import darksnail.gemma4
importlib.reload(darksnail.arc3)
importlib.reload(darksnail.gemma4)
importlib.reload(darksnail)

from darksnail import (
    Arc3AffectiveAmygdalaRouter,
    Arc3EpisodicBeltramiShell,
    Arc3HRMRecurrentCore,
    Arc3TransitionRecord,
    DarkSnailArc3Agent,
    DarkSnailArc3WorldModel,
    Gemma4BeltramiCallosalShell,
    HRMRecurrentBlock,
    MagicNorm,
    SlowFastTriadCascade,
    TemporalClockContentSplitter,
    TransitionRecord,
    ZenoAmygdalaController,
    attach_gemma4_beltrami_hook,
    compute_topological_symmetry_potential,
    extract_arc3_beltrami_medoids,
    load_gemma4_router_sidecar,
    patch_gemma4_model,
    verify_pointwise_orthogonality,
)
print(f"[OK] Imported darksnail v{darksnail.__version__} with DarkSnailArc3Agent, HRM/TRM, Amygdala & Gemma 4 31B integration.")

## 2. Mathematical Invariants Gate (`TemporalClockContentSplitter`, $\mathcal{G}_{\text{act}} / \Omega_{\text{reg}}$ Factorization, `Arc3HRMRecurrentCore`, `Arc3AffectiveAmygdalaRouter` & `SlowFastTriadCascade`)

In [ ]:
# === CELL 2: Verify Mathematical Invariants of DarkSnailArc3Agent ===
torch.manual_seed(1729)
np.random.seed(1729)

# 1. [Gate 2a] TemporalClockContentSplitter: Zero Spatial Border Priors, Interior Bar Masking & Cross-Level Persistence
splitter = TemporalClockContentSplitter()
f0 = np.full((64, 64), 4, dtype=np.int32)
# Place an exogenous 1x20 timer bar right in the CENTER of the grid (row 30, cols 10..30)
f0[30, 10:30] = 9
f0[35:37, 50:52] = 6   # 4px life counter icon
f0[12:16, 20:24] = 14  # 4x4 controllable sprite
splitter.initialize_level(f0)

# Action 4 moves the agent sprite and depletes 1px of the interior timer at (30, 29)
f1 = f0.copy()
f1[12:16, 20:24] = 4
f1[12:16, 24:28] = 14
f1[30, 29] = 5
exp1, nope_d1, reset1 = splitter.observe_transition(f0, f1, action_key=4)

# Action 1 hits a wall: only the interior timer bar depletes
f2 = f1.copy()
f2[30, 20:29] = 5
exp2, nope_d2, reset2 = splitter.observe_transition(f1, f2, action_key=1)

# Level phase reset: agent returns to f0, timer refills, 4px life icon depletes
f3 = f0.copy()
f3[35:37, 50:52] = 4
exp3, nope_d3, reset3 = splitter.observe_transition(f2, f3, action_key=4)

interior_masked = int(splitter.clock_mask[30, 10:30].sum())
hash_f1 = splitter.hash_nope(f1)
hash_f2 = splitter.hash_nope(f2)
splitter.initialize_level(f0, preserve_clock_mask=True)
preserved_masked = int(splitter.clock_mask[30, 10:30].sum())
print(f"[Gate 2a | TemporalClockContentSplitter] Interior [30, 10:30] masked px: {interior_masked}/20 | "
      f"Wall-collision nope hash equality: {hash_f1 == hash_f2} | Preserved across level: {preserved_masked}/20")
assert interior_masked == 20 and hash_f1 == hash_f2 and reset3 is True and preserved_masked == 20

# 2. [Gate 2b] extract_arc3_beltrami_medoids: Hollow Ring Interior Guarantee & Dyadic Quadtree Subdivision
grid_ring = np.zeros((64, 64), dtype=np.int32)
grid_ring[10:21, 10:21] = 4
grid_ring[12:19, 12:19] = 0  # Hollow center at (15, 15) is background 0
cands_ring = extract_arc3_beltrami_medoids(grid_ring, max_candidates=16)
ring_medoids = [c for c in cands_ring if c["color"] == 4]
assert len(ring_medoids) >= 1 and grid_ring[ring_medoids[0]["y"], ring_medoids[0]["x"]] == 4

grid_tiles = np.zeros((64, 64), dtype=np.int32)
for tr in range(3):
    for tc in range(3):
        r0, c0 = 36 + tr * 8, 36 + tc * 8
        grid_tiles[r0 : r0 + 6, c0 : c0 + 6] = 8 if (tr + tc) % 2 == 0 else 9
cands_tiles = extract_arc3_beltrami_medoids(grid_tiles, max_candidates=48)
quadtree_hits = sum(
    1 for tr in range(3) for tc in range(3)
    if any(36 + tr * 8 <= c["y"] < 36 + tr * 8 + 6 and 36 + tc * 8 <= c["x"] < 36 + tc * 8 + 6 for c in cands_tiles)
)
print(f"[Gate 2b | Dyadic Quadtree Medoids] Hollow-ring medoid color={grid_ring[ring_medoids[0]['y'], ring_medoids[0]['x']]} (non-zero!) | "
      f"3x3 tile sub-panel coverage: {quadtree_hits}/9 tiles")
assert quadtree_hits >= 6

# 3. [Gate 2c] Topological Betti (beta_0, beta_1) & Dihedral D_4 Symmetry Potential
sym_grid = np.zeros((64, 64), dtype=np.int32)
sym_grid[16:48, 16:48] = 2
sym_grid[24:40, 24:40] = 0
asym_grid = sym_grid.copy()
asym_grid[18:25, 42:47] = 5
phi_sym = compute_topological_symmetry_potential(sym_grid)
phi_asym = compute_topological_symmetry_potential(asym_grid)
print(f"[Gate 2c | Topological Symmetry Potential] Phi_topo(symmetric)={phi_sym:.4f} > Phi_topo(asymmetric)={phi_asym:.4f}")
assert phi_sym > phi_asym

# 4. [Gate 2d] Non-Local Causal Operator Factorization (G_act vs Omega_reg) & Dihedral D_4 Shape Transfer
wm_op = DarkSnailArc3WorldModel()
f_act0 = np.zeros((64, 64), dtype=np.int32)
f_act0[29:31, 2:10] = 11   # Peripheral 2x8 horizontal actuator at (6, 30)
f_act0[20:24, 20:26] = 3   # Passive 4x6 state register at [20:24, 20:26]
f_act1 = f_act0.copy()
f_act1[20:24, 20:26] = 8   # Clicking (6, 30) mutates remote passive register (24 px, L_inf > 4)
wm_op.record_transition(
    f_act0, f_act1, action_key=(6, 6, 30),
    candidate_meta={"x": 6, "y": 30, "kind": "cc_object", "area": 16, "bbox": (29, 2, 30, 9)},
)
d4_rot_prior = wm_op.get_d4_shape_prior({"x": 30, "y": 6, "kind": "cc_object", "area": 16, "bbox": (2, 29, 9, 30)})
print(f"[Gate 2d | Non-Local Operator & D_4 Transfer] (6,30) in G_act: {(6, 30) in wm_op.remote_actuators} | "
      f"Passive (22,22) pruned: {wm_op.is_driven_passive_register(22, 22)} | 90-deg D_4 prior: {d4_rot_prior:.2f}")
assert (6, 30) in wm_op.remote_actuators and wm_op.is_driven_passive_register(22, 22) is True and d4_rot_prior > 0.4

# 5. [Gate 2e] Arc3HRMRecurrentCore, Arc3AffectiveAmygdalaRouter & SlowFastTriadCascade Invariants
wm = DarkSnailArc3WorldModel(d_reasoner=512, d_shell=8, unroll_steps=4)
shell = Arc3EpisodicBeltramiShell(d_model=512, target_tokens=32, use_gemma4_shell=True)
amygdala = Arc3AffectiveAmygdalaRouter(d_model=512)
for step_i in range(6):
    f_step = np.roll(sym_grid, shift=step_i, axis=0)
    z_H, s_diag = shell(f_step, prev_frame_2d=sym_grid, action_id=(step_i % 4) + 1, return_diagnostics=True)
z_L = shell._history_tokens[-1]
amyg_mod = amygdala.step(z_H, z_L, last_nope_delta=0, is_new_state=False, delta_phi=-0.05, untried_ratio=0.9, consecutive_non_progress=6, step_idx=6)
priors, cas_diag = wm.compute_triad_action_priors(z_H, z_L, unroll_steps=4, hrm_steps=amyg_mod["reasoning_steps"])
print(f"[Gate 2e | HRM + Amygdala + SlowFastTriadCascade] Max Radial Drift: {cas_diag['max_norm_drift']:.3e} | "
      f"Action Triad Ortho Err: {cas_diag['action_triad_orthogonality_error']:.3e} | "
      f"HRM Steps: {cas_diag['hrm_steps_taken']} | Amygdala(V={amyg_mod['valence']:.2f}, U={amyg_mod['uncertainty']:.2f}, Omega={amyg_mod['urgency']:.2f}) | "
      f"Callosal Escalate: {amyg_mod['should_call_gemma4_system2']}")
assert cas_diag["max_norm_drift"] == 0.0
assert cas_diag["max_orthogonality_error"] <= 1e-8
assert cas_diag["action_triad_orthogonality_error"] <= 1e-8
assert amyg_mod["should_call_gemma4_system2"] is True

## 3. Live `google/gemma-4-31B-it` Backbone, `gemma4_31b_router_v4.safetensors` Sidecar & Dual-Pathway Layer-29 `attach_gemma4_system2`

In [ ]:
# === CELL 3: Load google/gemma-4-31B-it + Router V4 Sidecar & Attach Layer-29 System-2 Hook ===
BACKBONE_MODEL = os.environ.get("DARKSNAIL_BACKBONE", "google/gemma-4-31B-it")
SIDECAR_PATH = Path("weights/gemma4_31b_router_v4.safetensors")
lh_model = None
tokenizer = None

if device == "cuda" and hf_token and vram_gb >= 14.0:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    load_kwargs = {"device_map": "auto", "token": hf_token}
    if vram_gb < 68.0:
        try:
            from transformers import BitsAndBytesConfig
            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            print(f"[VRAM={vram_gb:.1f} GB < 68 GB] Loading {BACKBONE_MODEL} in 4-bit NF4 (bfloat16 compute)...")
        except Exception as exc:
            print(f"[WARN] bitsandbytes unavailable ({exc}); attempting bfloat16 load...")
            load_kwargs["torch_dtype"] = torch.bfloat16
    else:
        load_kwargs["torch_dtype"] = torch.bfloat16
        print(f"Loading {BACKBONE_MODEL} in native bfloat16 on {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM)...")

    tokenizer = AutoTokenizer.from_pretrained(BACKBONE_MODEL, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    lh_model = AutoModelForCausalLM.from_pretrained(BACKBONE_MODEL, **load_kwargs)
    lh_model.eval()

    # Patch the 10 Global Full-Attention layers [5, 11, 17, 23, 29, 35, 41, 47, 53, 59] with SurgicalGemma4Attention
    patch_gemma4_model(lh_model, enable_global_hook=True)
    if SIDECAR_PATH.exists():
        loaded_routers = load_gemma4_router_sidecar(lh_model, SIDECAR_PATH)
        print(f"[OK] Loaded {loaded_routers}/10 Global SurgicalGemma4Attention routers from {SIDECAR_PATH}")

    # Verify attach_gemma4_system2 and Dual-Pathway deliberate_with_gemma4_system2 on DarkSnailArc3Agent
    demo_agent = DarkSnailArc3Agent(max_actions=100, seed=0, use_gemma4_shell=True)
    g4_shell = demo_agent.attach_gemma4_system2(lh_model, tokenizer=tokenizer, hook_layer=29)
    print(f"[OK] Attached Gemma4BeltramiCallosalShell at Global Layer {g4_shell.hook_layer} (Dual-Pathway 5376 -> 4096 -> 512 + Symbolic Logit Priors active)")
else:
    print("[INFO] Live Gemma 4 31B load skipped (requires CUDA GPU >= 14 GB + HF_TOKEN).")
    if SIDECAR_PATH.exists():
        print(f"[OK] Verified router sidecar present at {SIDECAR_PATH} ({SIDECAR_PATH.stat().st_size / 1024:.1f} KB).")

## 4. Interactive Single-Game Telemetry (`ls20` Fuel-Clock Navigation, `vc33` Hydraulic Puzzle, `sc25` Multi-Level Actuator & `lp85` Stepper)

In [ ]:
# === CELL 4: Interactive Single-Game Telemetry (ls20, vc33, sc25, lp85) ===
from arc_agi import Arcade, OperationMode

arc = Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=str(env_dir))
showcase_runs = {}

for gid, budget in [("ls20", 220), ("vc33", 100), ("sc25", 600), ("lp85", 50)]:
    env = arc.make(gid)
    agent = DarkSnailArc3Agent(max_actions=budget, seed=0)
    if lh_model is not None and tokenizer is not None:
        agent.attach_gemma4_system2(lh_model, tokenizer=tokenizer, hook_layer=29)
    res = agent.play_game(env, max_actions=budget)
    showcase_runs[gid] = (agent, res)
    print(
        f"[{gid}] levels_completed={res['levels_completed']}/{res['win_levels']} | "
        f"actions={res['actions_taken']} | clock_px_masked={res['clock_pixels_masked']} | "
        f"remote_actuators={res.get('remote_actuators_discovered', 0)} | "
        f"gemma4_calls={res.get('gemma4_calls_made', 0)} | "
        f"completion_steps={res['level_completion_steps']}"
    )

## 5. Full 25-Game ARC-AGI-3 Offline Evaluation Suite (`random_legal` vs. `unshielded_greedy` vs. `darksnail_v2_arc3`)

In [ ]:
# === CELL 5: Full 25-Game ARC-AGI-3 Offline Benchmark & Paired Statistical Tests ===
import json
from scripts.evaluate_darksnail_arc3 import evaluate_condition, compute_paired_stats, PUBLIC_25_GAME_IDS

# Set RUN_FRESH_BENCHMARK = True to re-run all 25 games x 3 conditions live,
# or False to inspect the pre-computed 1000-action evaluation summary first.
RUN_FRESH_BENCHMARK = False
MAX_ACTIONS_PER_GAME = int(os.environ.get("ARC3_MAX_ACTIONS", "1000"))
SUMMARY_PATH = Path("results/darksnail_arc3_evaluation_summary.json")

if RUN_FRESH_BENCHMARK or not SUMMARY_PATH.exists():
    cond_results = {}
    for cond in ("random_legal", "unshielded_greedy", "darksnail_v2_arc3"):
        print(f"\n=== Evaluating Condition: {cond} (max_actions={MAX_ACTIONS_PER_GAME}) ===")
        cond_results[cond] = evaluate_condition(
            condition_name=cond,
            env_dir=env_dir,
            game_ids=PUBLIC_25_GAME_IDS,
            max_actions=MAX_ACTIONS_PER_GAME,
            seed=0,
            lh_model=lh_model,
            tokenizer=tokenizer,
        )
    ds_levels = [float(g["levels_completed"]) for g in cond_results["darksnail_v2_arc3"]["per_game"]]
    ug_levels = [float(g["levels_completed"]) for g in cond_results["unshielded_greedy"]["per_game"]]
    rnd_levels = [float(g["levels_completed"]) for g in cond_results["random_legal"]["per_game"]]
    ds_rhae = [float(g["official_rhae_score"]) for g in cond_results["darksnail_v2_arc3"]["per_game"]]
    ug_rhae = [float(g["official_rhae_score"]) for g in cond_results["unshielded_greedy"]["per_game"]]
    rnd_rhae = [float(g["official_rhae_score"]) for g in cond_results["random_legal"]["per_game"]]
    eval_summary = {
        "benchmark": "ARC-AGI-3 Public Offline Suite (25 Games, 183 Levels)",
        "max_actions_per_game": MAX_ACTIONS_PER_GAME,
        "seed": 0,
        "conditions": cond_results,
        "paired_statistics": {
            "darksnail_vs_random_levels_completed": compute_paired_stats(ds_levels, rnd_levels),
            "darksnail_vs_unshielded_greedy_levels_completed": compute_paired_stats(ds_levels, ug_levels),
            "darksnail_vs_random_rhae_score": compute_paired_stats(ds_rhae, rnd_rhae),
            "darksnail_vs_unshielded_greedy_rhae_score": compute_paired_stats(ds_rhae, ug_rhae),
        },
    }
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_summary, f, indent=2)
else:
    with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
        eval_summary = json.load(f)

print(f"Benchmark: {eval_summary['benchmark']} (max_actions={eval_summary['max_actions_per_game']}, seed={eval_summary['seed']})\n")
print(f"{'Condition':22s} | {'Levels Solved':14s} | {'Level %':8s} | {'Games >= 1 Lvl':15s} | {'Official RHAE %':15s}")
print("-" * 86)
for cname in ("random_legal", "unshielded_greedy", "darksnail_v2_arc3"):
    c = eval_summary["conditions"][cname]
    print(
        f"{cname:22s} | "
        f"{c['total_levels_completed']:3d} / {c['total_levels_available']:3d}      | "
        f"{c['level_completion_rate_pct']:6.2f}%  | "
        f"{c['games_with_at_least_1_level_completed']:2d} / {c['num_games']:2d} ({c['games_with_at_least_1_level_pct']:4.1f}%) | "
        f"{c['official_rhae_scorecard_score_pct']:13.4f}%"
    )

ps = eval_summary["paired_statistics"]["darksnail_vs_unshielded_greedy_levels_completed"]
print("\nPaired Non-Parametric Statistical Comparison (darksnail_v2_arc3 vs. Baselines across N=25 Games):")
print(f"  - Wins / Losses / Ties          : {ps['wins']} / {ps['losses']} / {ps['ties']}")
print(f"  - Mean Difference (Levels/Game) : +{ps['mean_diff']:.2f} (10,000-Sample Bootstrap 95% CI: [{ps['bootstrap_95_ci'][0]:.2f}, {ps['bootstrap_95_ci'][1]:.2f}])")
print(f"  - Exact Binomial Sign Test p    : {ps['sign_test_p_value']:.6f}")
print(f"  - Wilcoxon Signed-Rank Test p   : {ps['wilcoxon_p_value']:.6f}")

print("\nSolved Games Breakdown (darksnail_v2_arc3):")
for g in eval_summary["conditions"]["darksnail_v2_arc3"]["per_game"]:
    if g["levels_completed"] > 0:
        print(
            f"  * {g['game_id']:14s}: {g['levels_completed']}/{g['win_levels']} levels | "
            f"steps={g['level_completion_steps']} | clock_px_masked={g['clock_pixels_masked']} | "
            f"remote_act={g.get('remote_actuators_discovered', 0)} | RHAE={g['official_rhae_score']:.4f}%"
        )

## 6. Visual Diagnostic Gallery (`TemporalClockContentSplitter` $\mathcal{H}_{\text{clock}}$ Mask & Dyadic Quadtree Medoids)

In [ ]:
# === CELL 6: Visualize Exogenous Clock Masking & Dyadic Quadtree Medoid Extraction ===
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

ARC3_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00", "#AAAAAA", "#F012BE", "#FF851B",
    "#7FDBFF", "#870C25", "#39CCCC", "#B10DC9", "#01FF70", "#85144b", "#3D9970", "#FFFFFF",
]
arc3_cmap = ListedColormap(ARC3_COLORS)

fig, axes = plt.subplots(len(showcase_runs), 3, figsize=(13, 4.2 * len(showcase_runs)))
if len(showcase_runs) == 1:
    axes = [axes]

for row_idx, (gid, (agent, res)) in enumerate(showcase_runs.items()):
    init_hash = agent.world_model.initial_state_hash
    frame_2d = agent.world_model.state_frames.get(init_hash, agent.prev_frame_2d)
    clock_mask = agent.world_model.splitter.clock_mask
    medoids = extract_arc3_beltrami_medoids(frame_2d, max_candidates=36, clock_mask=clock_mask)

    ax0, ax1, ax2 = axes[row_idx]
    ax0.imshow(frame_2d, cmap=arc3_cmap, vmin=0, vmax=15)
    ax0.set_title(f"{gid}: Raw 64x64 Frame (Lvls={res['levels_completed']}/{res['win_levels']})")
    ax0.axis("off")

    ax1.imshow(frame_2d, cmap=arc3_cmap, vmin=0, vmax=15, alpha=0.45)
    if clock_mask is not None:
        ax1.imshow(np.ma.masked_where(~clock_mask, clock_mask), cmap="cool", vmin=0, vmax=1)
    ax1.set_title(f"{gid}: Exogenous H_clock Mask ({res['clock_pixels_masked']} px)")
    ax1.axis("off")

    ax2.imshow(frame_2d, cmap=arc3_cmap, vmin=0, vmax=15)
    xs = [m["x"] for m in medoids]
    ys = [m["y"] for m in medoids]
    ax2.scatter(xs, ys, c="cyan", edgecolors="black", s=45, linewidths=1.0, label="Beltrami Medoids")
    ax2.set_title(f"{gid}: Top-{len(medoids)} Dyadic Beltrami Medoids")
    ax2.axis("off")

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/darksnail_arc3_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Saved diagnostic figure to results/darksnail_arc3_diagnostics.png")